# The Price is Right

## 第八周课程安排

第一天：Modal.com 和 SpecialistAgent  
第二天：RAG、FrontierAgent、Ensemble Agent  
第三天：ScannerAgent、MessengerAgent  
第四天：AutonomousPlannerAgent 和 DealAgentFramework  
第五天：The Price Is Right 终章


今天我们将构建拼图的另一块：一个通过订阅 RSS 源来寻找有前景优惠的 ScanningAgent。

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5.1'

In [5]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|█████████████████████████████████████| 3/3 [01:40<00:00, 33.35s/it]


In [6]:
len(deals)

30

In [7]:
deals[10].describe()

'Title: Adjustable Metal Laptop Stand for $14 + free shipping w/ Prime\nDetails: This laptop stand is at its lowest price on Amazon at $14.24, down from the regular $16.99 price. It folds flat for travel and holds up to 17.6 lbs., with ventilation holes built in to help cool a laptop during use. Shipping is free for Prime members.\xa0 Buy Now at Amazon\nFeatures: Adjustable height and angle for ergonomic use Metal construction supports up to 17.6 lbs. Rubber and silicone pads protect and secure laptops Ventilated top design for heat dissipation Foldable, portable design for travel Fits laptops from 10" to 15.6"\nURL: https://www.dealnews.com/Adjustable-Metal-Laptop-Stand-for-14-free-shipping-w-Prime/22166089.html?iref=rss-c39'

### 我们将让 GPT-5.1 总结优惠信息并识别其价格

In [8]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [9]:
# 根据抓取的优惠信息创建合适的 user prompt

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [10]:
# 让我们为刚刚抓取的优惠信息创建一个 user prompt，并看看它的开头

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Mersoco Vintage Wood Bluetooth Turntable w/ Built-in Speakers for $30 + free shipping w/ Prime
Details: This Mersoco turntable is $10 off its regular $39.99 price at Amazon, matching its all-time low. It plays records at all three common speeds and works with 7", 10", and 12" vinyl, plus it connects over Bluetooth or via RCA and headphone outputs for flexible listening options.

In [14]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="low")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='This 65-inch Hisense M7 is a 4K UHD Google Smart TV featuring a mini-LED QLED panel with full array local dimming for improved black levels and contrast. It supports up to 1000 nits of peak brightness, making HDR content pop in both dark and bright rooms. Motion Rate 240 helps smooth out fast action in sports and movies, while a built-in subwoofer with Dolby Atmos delivers more immersive sound. It also supports a wide range of HDR formats including Dolby Vision, HDR10+, HDR10, and HLG for broad content compatibility.', price=374.0, url='https://www.dealnews.com/Hisense-M7-65-Mini-LED-QLED-4-K-UHD-Google-Smart-TV-for-374-free-shipping/22166038.html?iref=rss-c142'), Deal(product_description='The Rachio 3 12-Zone WiFi Smart Sprinkler Controller manages up to 12 irrigation zones and connects to your home network for app-based control. Using real-time weather data, it automatically skips watering during rain, high wind, or freezing temperatures

In [15]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


This 65-inch Hisense M7 is a 4K UHD Google Smart TV featuring a mini-LED QLED panel with full array local dimming for improved black levels and contrast. It supports up to 1000 nits of peak brightness, making HDR content pop in both dark and bright rooms. Motion Rate 240 helps smooth out fast action in sports and movies, while a built-in subwoofer with Dolby Atmos delivers more immersive sound. It also supports a wide range of HDR formats including Dolby Vision, HDR10+, HDR10, and HLG for broad content compatibility.
374.0
https://www.dealnews.com/Hisense-M7-65-Mini-LED-QLED-4-K-UHD-Google-Smart-TV-for-374-free-shipping/22166038.html?iref=rss-c142

The Rachio 3 12-Zone WiFi Smart Sprinkler Controller manages up to 12 irrigation zones and connects to your home network for app-based control. Using real-time weather data, it automatically skips watering during rain, high wind, or freezing temperatures to help conserve water. Its Yard Map feature lets you create a visual layout of your lan

In [16]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [17]:
from agents.scanner_agent import ScannerAgent

In [24]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [25]:
result

DealSelection(deals=[Deal(product_description='This 65-inch Hisense M7 TV features a Mini-LED QLED panel with full array local dimming for deep blacks and enhanced contrast, paired with 4K UHD resolution and AI 4K upscaling to sharpen lower-resolution content. It reaches up to 1,000 nits of peak brightness, making HDR highlights stand out even in brighter rooms, and supports Dolby Vision, HDR10+, HDR10, and HLG for wide compatibility with modern video formats. Motion Rate 240 helps smooth fast-moving sports and action scenes, while the built-in subwoofer with Dolby Atmos support delivers more immersive, cinema-like sound. Running Google Smart TV, it offers easy access to streaming apps and voice control integration in a single, large-screen package.', price=374.0, url='https://www.dealnews.com/Hisense-M7-65-Mini-LED-QLED-4-K-UHD-Google-Smart-TV-for-374-free-shipping/22166038.html?iref=rss-c142'), Deal(product_description='Designed specifically for the iPhone 16 Pro Max, this smart disp

### 介绍 Pushover

Pushover 是一个向手机发送推送通知的实用工具。

设置和安装非常简单！

只需访问 https://pushover.net/ 并点击右上角的 "Login or Signup" 注册免费账户，然后创建你的 API 密钥。

注册后，在主页上点击 "Create an Application/API Token"，给它起个名字（如 AIEngineer）并点击 Create Application。

然后在你的 `.env` 文件中添加 2 行：

PUSHOVER_USER=_放在 Pushover 主页右上角的密钥，可能以 u 开头_  
PUSHOVER_TOKEN=_点击进入你的新应用（Agents 或其他名称）后的密钥，可能以 a 开头_

记得保存你的 `.env` 文件，保存后运行 `load_dotenv(override=True)` 来设置环境变量。

最后，点击 "Add Phone, Tablet or Desktop" 在手机上安装。

In [26]:
load_dotenv(override=True)

True

In [27]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [28]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [29]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [30]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [31]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and GPT-5.1
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [32]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using GPT-5.1 to craft the message
16:09:36 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= gpt-5.1; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5.1; provider = openai
INFO:openai._base_client:Retrying request to /chat/completions in 0.402974 seconds
16:10:44 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
